# Crop Recommendation — Model Training Pipeline

Predicts the best crop (`label`) from 7 soil/climate features:
`N`, `P`, `K`, `temperature`, `humidity`, `ph`, `rainfall`.

**Steps:** load & validate → split → preprocess → train & compare models →
evaluate → tune best model → feature importance → save artifacts → inference.

> Run cells top to bottom (`Runtime > Run all` in Colab). Plots render inline.

## 0. Setup
Upload `Crop_recommendation.csv` when prompted (or mount Google Drive and edit the path below).

In [ ]:
# If running in Colab, upload the CSV file
try:
    from google.colab import files
    import os
    if not os.path.exists("Crop_recommendation.csv"):
        print("Please upload Crop_recommendation.csv")
        uploaded = files.upload()
except ImportError:
    print("Not running in Colab — make sure Crop_recommendation.csv is in the working directory.")


In [ ]:
!pip install -q xgboost scikit-learn seaborn joblib

In [ ]:
import json
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

RANDOM_SEED = 42
DATA_PATH = "Crop_recommendation.csv"
FEATURE_COLUMNS = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
TARGET_COLUMN = "label"

sns.set_style("whitegrid")
np.random.seed(RANDOM_SEED)


## 1. Load and validate the data
Check dtypes, missing values, and class balance.

In [ ]:
def load_and_validate(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Column check
    expected_cols = set(FEATURE_COLUMNS + [TARGET_COLUMN])
    missing_cols = expected_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"Dataset is missing expected columns: {missing_cols}")

    # Dtype check
    non_numeric = [c for c in FEATURE_COLUMNS if not pd.api.types.is_numeric_dtype(df[c])]
    if non_numeric:
        raise TypeError(f"Expected numeric dtype for columns: {non_numeric}")

    # Missing values check
    null_counts = df[FEATURE_COLUMNS + [TARGET_COLUMN]].isnull().sum()
    if null_counts.sum() > 0:
        raise ValueError(f"Dataset contains missing values:\n{null_counts[null_counts > 0]}")

    # Class balance
    class_counts = df[TARGET_COLUMN].value_counts()
    print(f"Loaded {len(df)} rows, {df[TARGET_COLUMN].nunique()} classes.")
    print(f"Class balance -> min: {class_counts.min()}, max: {class_counts.max()}, "
          f"mean: {class_counts.mean():.1f}")
    if class_counts.min() != class_counts.max():
        print("Warning: classes are not perfectly balanced.")
    else:
        print("Classes are perfectly balanced.")

    return df

df = load_and_validate(DATA_PATH)
df.head()


In [ ]:
df.describe()

In [ ]:
# Visualize class balance
fig, ax = plt.subplots(figsize=(12, 5))
df[TARGET_COLUMN].value_counts().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Samples per Crop Class")
ax.set_ylabel("Count")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## 2. Train/test split
80/20 split, stratified by `label`, fixed random seed for reproducibility.

In [ ]:
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y,
)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")


## 3. Preprocess
Scale numeric features with `StandardScaler`, encode `label` with `LabelEncoder`.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

print("Classes:", list(label_encoder.classes_))


## 4. Train and compare multiple models
Random Forest, Logistic Regression, SVM (RBF), KNN, and Gradient Boosting.

In [ ]:
candidate_models = {
    "RandomForest": RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200),
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
    "SVM_RBF": SVC(kernel="rbf", random_state=RANDOM_SEED, probability=True),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_SEED),
}

trained_models = {}
for name, model in candidate_models.items():
    model.fit(X_train_scaled, y_train_enc)
    trained_models[name] = model
    print(f"Trained: {name}")


## 5. Evaluate each model
Accuracy, macro-averaged precision/recall/F1, and a confusion matrix per model.

In [ ]:
def evaluate_model(model, model_name):
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test_enc, y_pred)
    prec_macro = precision_score(y_test_enc, y_pred, average="macro", zero_division=0)
    rec_macro = recall_score(y_test_enc, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_test_enc, y_pred, average="macro", zero_division=0)

    print(f"--- {model_name} ---")
    print(f"Accuracy:        {acc:.4f}")
    print(f"Macro Precision: {prec_macro:.4f}")
    print(f"Macro Recall:    {rec_macro:.4f}")
    print(f"Macro F1:        {f1_macro:.4f}\n")

    return {
        "model_name": model_name,
        "accuracy": acc,
        "precision_macro": prec_macro,
        "recall_macro": rec_macro,
        "f1_macro": f1_macro,
        "y_pred": y_pred,
    }

all_results = [evaluate_model(model, name) for name, model in trained_models.items()]


In [ ]:
# Confusion matrix for each model (inline)
for result in all_results:
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay.from_predictions(
        y_test_enc, result["y_pred"],
        display_labels=label_encoder.classes_,
        xticks_rotation=90,
        cmap="Blues",
        ax=ax,
        colorbar=False,
    )
    ax.set_title(f"Confusion Matrix — {result['model_name']}")
    plt.tight_layout()
    plt.show()


In [ ]:
# Side-by-side model comparison chart
results_df = pd.DataFrame(all_results)[
    ["model_name", "accuracy", "precision_macro", "recall_macro", "f1_macro"]
].set_index("model_name")

ax = results_df.plot(kind="bar", figsize=(10, 6), ylim=(0, 1.05))
ax.set_title("Model Comparison (Test Set)")
ax.set_ylabel("Score")
plt.xticks(rotation=30, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

results_df


## 6. Hyperparameter tuning
Tune the best baseline model (Random Forest) with `GridSearchCV` + stratified 5-fold CV.

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_SEED),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train_scaled, y_train_enc)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV macro-F1: {grid_search.best_score_:.4f}")

tuned_model = grid_search.best_estimator_
tuned_result = evaluate_model(tuned_model, "RandomForest_Tuned")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test_enc, tuned_result["y_pred"],
    display_labels=label_encoder.classes_,
    xticks_rotation=90,
    cmap="Greens",
    ax=ax,
    colorbar=False,
)
ax.set_title("Confusion Matrix — RandomForest (Tuned)")
plt.tight_layout()
plt.show()


## 7. Feature importance
Which soil/climate factors matter most for the tuned Random Forest?

In [ ]:
importances = tuned_model.feature_importances_
order = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=importances[order], y=np.array(FEATURE_COLUMNS)[order], ax=ax, color="seagreen")
ax.set_title("Feature Importance (Tuned Random Forest)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

for name, score in sorted(zip(FEATURE_COLUMNS, importances), key=lambda x: x[1], reverse=True):
    print(f"{name:15s} {score:.4f}")


## 8. Save the final model, scaler, and label encoder
Saved with `joblib` so they can be reloaded for inference later (e.g. download from Colab or push to Drive).

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(tuned_model, "models/crop_model.joblib")
joblib.dump(scaler, "models/scaler.joblib")
joblib.dump(label_encoder, "models/label_encoder.joblib")

print("Saved: models/crop_model.joblib, models/scaler.joblib, models/label_encoder.joblib")

# Optional: download to your machine if running in Colab
try:
    from google.colab import files
    files.download("models/crop_model.joblib")
    files.download("models/scaler.joblib")
    files.download("models/label_encoder.joblib")
except ImportError:
    pass


## 9. Inference function
Predict the recommended crop for a new set of readings.

In [ ]:
def predict_crop(N, P, K, temperature, humidity, ph, rainfall,
                  model=None, scaler=None, label_encoder=None):
    """Predict the recommended crop for a single set of soil/climate readings."""
    if model is None:
        model = joblib.load("models/crop_model.joblib")
    if scaler is None:
        scaler = joblib.load("models/scaler.joblib")
    if label_encoder is None:
        label_encoder = joblib.load("models/label_encoder.joblib")

    input_df = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]], columns=FEATURE_COLUMNS)
    input_scaled = scaler.transform(input_df)
    pred_encoded = model.predict(input_scaled)[0]
    pred_label = label_encoder.inverse_transform([pred_encoded])[0]

    top3 = None
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(input_scaled)[0]
        top3_idx = np.argsort(proba)[::-1][:3]
        top3 = [(label_encoder.inverse_transform([i])[0], round(float(proba[i]), 4)) for i in top3_idx]

    return pred_label, top3


# Demo: predict on the first test-set example
sample = X_test.iloc[0]
pred_label, top3 = predict_crop(
    sample["N"], sample["P"], sample["K"],
    sample["temperature"], sample["humidity"], sample["ph"], sample["rainfall"],
    model=tuned_model, scaler=scaler, label_encoder=label_encoder,
)

print("Input:", sample.to_dict())
print("Actual label:   ", y_test.iloc[0])
print("Predicted label:", pred_label)
print("Top-3 candidates:", top3)


In [ ]:
# Try your own values here
predict_crop(
    N=90, P=42, K=43,
    temperature=20.9, humidity=82.0, ph=6.5, rainfall=202.9,
    model=tuned_model, scaler=scaler, label_encoder=label_encoder,
)
